# Online Retail II - Exploration et nettoyage

## Chargement des données

In [2]:
import pandas as pd

In [3]:
df = pd.read_csv("../data/raw/online_retail_II.csv") 

In [9]:
df.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


### Vérifions le nombre de lignes et de colonnes de notre dataset

In [6]:
df.shape

(1067371, 8)

- Notre dataset compte exctement 1067371 ligne et 8 colonnes

## Audit de qualité

### affichons les types de colonnes et le nombre de valeurs non nulles

In [10]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1067371 entries, 0 to 1067370
Data columns (total 8 columns):
 #   Column       Non-Null Count    Dtype  
---  ------       --------------    -----  
 0   Invoice      1067371 non-null  str    
 1   StockCode    1067371 non-null  str    
 2   Description  1062989 non-null  str    
 3   Quantity     1067371 non-null  int64  
 4   InvoiceDate  1067371 non-null  str    
 5   Price        1067371 non-null  float64
 6   Customer ID  824364 non-null   float64
 7   Country      1067371 non-null  str    
dtypes: float64(2), int64(1), str(5)
memory usage: 136.6 MB


Constats :
- `InvoiceDate` est de type texte, on va donc la convertir.
- `Customer ID` a beaucoup de valeurs manquantes.

### Valeurs manquantes

In [12]:
df.isnull().sum()

Invoice             0
StockCode           0
Description      4382
Quantity            0
InvoiceDate         0
Price               0
Customer ID    243007
Country             0
dtype: int64

Constats :
- Nous avons 6 colonnes qui n'ont pas de valeurs manquantes
- `Customer ID` a 243 007 valeurs manquantes ce qui fait environ 23% des lignes.
- `Description` a 4 382 valeurs manquantes (moins de 0,5 %).
- Impact : la question sur les clients ne portera que sur les lignes où `Customer ID` est renseigné.

### Comptons les doublons dans notre dataset

In [15]:
df.duplicated().sum()

np.int64(34335)

- 34 335 lignes sont des doublons exacts ce qui fait environ 3.2%.

### Statistiques descriptives

In [16]:
df.describe()

,Quantity,Price,Customer ID
count,1.067371e+06,1.067371e+06,824364.000000
mean,9.938898e+00,4.649388e+00,15324.638504
std,1.727058e+02,1.235531e+02,1697.464450
min,-8.099500e+04,-5.359436e+04,12346.000000
25%,1.000000e+00,1.250000e+00,13975.000000
50%,3.000000e+00,2.100000e+00,15255.000000
75%,1.000000e+01,4.150000e+00,16797.000000
max,8.099500e+04,3.897000e+04,18287.000000


- `Price` a un minimum négatif (-53 594,36), ce qui n'a pas de sens pour une vente.
- `Quantity` a un minimum négatif (-80 995) : retours ou annulations possibles
- Les maximums sont très éloignés de la médiane (`Quantity` : 80 995 contre 3 ; `Price` : 38 970 contre 2,10) : ce qui montre qu'il des valeurs extremes. Ces valeurs seront examinés.

### Quantités négatives

In [17]:
df[df["Quantity"] < 0].head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
178,C489449,22087,PAPER BUNTING WHITE LACE,-12,2009-12-01 10:33:00,2.95,16321.0,Australia
179,C489449,85206A,CREAM FELT EASTER EGG BASKET,-6,2009-12-01 10:33:00,1.65,16321.0,Australia
180,C489449,21895,POTTING SHED SOW 'N' GROW SET,-4,2009-12-01 10:33:00,4.25,16321.0,Australia
181,C489449,21896,POTTING SHED TWINE,-6,2009-12-01 10:33:00,2.10,16321.0,Australia
182,C489449,22083,PAPER CHAIN KIT RETRO SPOT,-12,2009-12-01 10:33:00,2.95,16321.0,Australia


- Nous constatons que toutes les factures des lignes où la quantité est négatives commencent par C (ex: C489449	).
    Ces retours ou annulations confirment donc que les quantitéd négatives sont des retours légitimes.

### Prix négatifs

In [26]:
df[df["Price"] < 0]

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
179403,A506401,B,Adjust bad debt,1,2010-04-29 13:36:00,-53594.36,NaN,United Kingdom
276274,A516228,B,Adjust bad debt,1,2010-07-19 11:24:00,-44031.79,NaN,United Kingdom
403472,A528059,B,Adjust bad debt,1,2010-10-20 12:04:00,-38925.87,NaN,United Kingdom
825444,A563186,B,Adjust bad debt,1,2011-08-12 14:51:00,-11062.06,NaN,United Kingdom
825445,A563187,B,Adjust bad debt,1,2011-08-12 14:52:00,-11062.06,NaN,United Kingdom


- Les factures correspondants aux lignes ou les prix sont négatifs commencent par A avec la description Adjust bad debt.
Ce ne sont donc pas des ventes et doivent etre évités lors du calcul du chiffre d'affaire.

### Résumé des problèmes ientifés

- InvoiceDate est en foromat texte, on devra le mettre en format date.
- Customer ID contient près de 23% de valeurs manquantes.
- Description : 4 382 valeurs manquantes (moins de 0,5 %).
- Doublons : 34 335 lignes identiques (environ 3,2 %), à analyser avant de supprimer.
- Retours : les quantités négatives viennent de factures commençant par C (annulations).
- Ajustements : les prix négatifs viennent de factures commençant par A (« Adjust bad debt »), ce ne sont pas des ventes.